# Projet Ralstonia solanacearum avec CobraPy

Dans ce notebook, je construis progressivement un modèle FBA/FVA pour *Ralstonia solanacearum* à partir d'un fichier SBML, en m'inspirant du tutoriel CobraPy mais en adaptant toutes les étapes à ce réseau.

## Question 1:  Charger (créer) le modèle SBML dans CobraPy

Je commence par importer les bibliothèques nécessaires et par créer l'objet 'model'
à partir du fichier SBML de l'espèce.


In [18]:
import cobra
from cobra.io import read_sbml_model
import pandas as pd
pd.options.display.max_rows = 100  # Pour afficher proprement les tableaux plus tard

# Je lis le fichier SBML de Ralstonia
model = read_sbml_model("ralsto_metexplore3.xml")

# Résumé brut de l'objet modèle pour vérifier que le chargement a fonctionné
model


Adding exchange reaction EX_hdcea_b with default bounds for boundary metabolite: hdcea_b.
Adding exchange reaction EX_RSc1944_b with default bounds for boundary metabolite: RSc1944_b.
Adding exchange reaction EX_3oochslac_b with default bounds for boundary metabolite: 3oochslac_b.
Adding exchange reaction EX_glu_D_b with default bounds for boundary metabolite: glu_D_b.
Adding exchange reaction EX_RipS4_b with default bounds for boundary metabolite: RipS4_b.
Adding exchange reaction EX_fecrm_b with default bounds for boundary metabolite: fecrm_b.
Adding exchange reaction EX_ni2_b with default bounds for boundary metabolite: ni2_b.
Adding exchange reaction EX_RipK_b with default bounds for boundary metabolite: RipK_b.
Adding exchange reaction EX_berb_b with default bounds for boundary metabolite: berb_b.
Adding exchange reaction EX_tartr_L_b with default bounds for boundary metabolite: tartr_L_b.
Adding exchange reaction EX_octa_b with default bounds for boundary metabolite: octa_b.
Addi

Name,_bc9d1403_3bf5_4994_8369_f3c28ce8995a
Memory address,7f4f9e736f00
Number of metabolites,2574
Number of reactions,3097
Number of genes,2219
Number of groups,253
Objective expression,0
Compartments,"boundary, extracellular, cytoplasm, periplasm"


## Question 2 : La taille du réseau; les réactions, métabolites, gènes

Comptage des composantes principales du modèle : nombre de réactions, nombre de métabolites et nombre de gènes.



In [19]:
nb_of_reactions = len(model.reactions)
nb_of_metabolites = len(model.metabolites)
nb_of_genes = len(model.genes)

print("Nombre de réactions   :", nb_of_reactions) # Le nombre de réactions dans model.reactions est “réseau original + quelques réactions d’échange auto-ajoutées”.
print("Nombre de métabolites :", nb_of_metabolites)
print("Nombre de gènes       :", nb_of_genes)


Nombre de réactions   : 3097
Nombre de métabolites : 2574
Nombre de gènes       : 2219


Notons que CobraPy ajoute automatiquement une réaction d’échange pour les métabolites marqués 'boundaryCondition = true' dans le SBML mais qui n’ont pas encore de réaction d’échange. Ce qui augmente le nombre total de réactions par rapport au fichier SBML initial.


## Question 3 : Indiquer la réaction de biomasse comme fonction objectif

Je recherche d'abord quelle est la réaction de biomasse: dans le modèle CobraPy, puis je la définis comme fonction objectif du modèle.


In [31]:
# Je retrouve la réactiona dont l'identifiant contient le mot "BIOMASS"
biomass_elements = []
for reaction in model.reactions: # je parcours toutes les réactions du modèle
    reactions_id = reaction.id.upper()
    
    if "BIOMASS" in reactions_id:
        biomass_elements.append(reaction.id)
        
biomass_elements


['BIOMASS']

In [33]:
# Je stocke l'ID de la réaction de biomasse trouvé ci-dessus
biomass_id = "BIOMASS"  

# Je définis cette réaction comme fonction objectif du modèle
model.objective = biomass_id

# Je teste si l'objectif est bien pris en compte par CobraPy en le demandant
model.objective.expression


1.0*BIOMASS - 1.0*BIOMASS_reverse_69053

La réaction BIOMASS représente la croissance (la synthèse de tous les constituants de la cellule), donc en la mettant comme objectif, nous maximisons le flux qui traverse cette réaction; ainsi donc, parmi toutes les façons possibles de faire fonctionner le réseau, on maximise celle où la cellule pousse le plus vite.